In [1]:
import sys, os, numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.gym_env import SnakeEnv
from core.env.enums import ObsType
from agents.q_learning import QLearningAgent

num_envs, total_episodes = 16, 50000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=15, height=15, obs_type=ObsType.VECTOR_11, num_apples=2, num_obstacles=15, seed=42 + i
        )
        for i in range(num_envs)
    ]
)

epsilon_decay = (0.01 / 1.0) ** (1 / total_episodes)
agent = QLearningAgent(state_dim=2048, action_dim=4, lr=0.1, gamma=0.95, epsilon_decay=epsilon_decay, seed=42)

training_logs, episode_rewards, completed = [], np.zeros(num_envs), 0
obs, infos = env.reset()

with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:

        actions = [agent.act(o) for o in obs]
        next_obs, rewards, terms, truncs, next_infos = env.step(actions)

        for i, (o, a, r, no, term, trunc) in enumerate(zip(obs, actions, rewards, next_obs, terms, truncs)):

            agent.update(o, a, r, no, term)
            episode_rewards[i] += r

            if term or trunc:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train()  # Decay epsilon per episode

                    training_logs.append({"episode": completed, "reward": episode_rewards[i], "epsilon": agent.epsilon})
                    if completed % 1000 == 0:
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (Shaped): {episode_rewards[i]:.2f} | Eps: {agent.epsilon:.3f}"
                        )
                episode_rewards[i] = 0

        obs = next_obs
        infos = next_infos

env.close()

Parallel Training:   2%|▏         | 1135/50000 [00:01<00:50, 965.80it/s]

Ep 1000/50000 | Avg Reward (Shaped): 38.55 | Eps: 0.912


Parallel Training:   4%|▍         | 2159/50000 [00:02<00:55, 862.52it/s]

Ep 2000/50000 | Avg Reward (Shaped): 39.60 | Eps: 0.832


Parallel Training:   6%|▌         | 3122/50000 [00:03<00:56, 827.49it/s]

...

Ep 48000/50000 | Avg Reward (Shaped): 682.75 | Eps: 0.012


Parallel Training:  98%|█████████▊| 49010/50000 [04:33<00:11, 89.37it/s] 

Ep 49000/50000 | Avg Reward (Shaped): 287.10 | Eps: 0.011


Parallel Training: 100%|██████████| 50000/50000 [04:45<00:00, 175.21it/s]

Ep 50000/50000 | Avg Reward (Shaped): 436.05 | Eps: 0.010


In [2]:
agent.save("q_learning_snake.pkl")

In [3]:
from core.utils import save_metrics

save_metrics(training_logs, "q_learning_training_logs.csv")